In [63]:
#!pip install pyspark

In [101]:

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("Spark starte")


Spark starte


In [106]:
#!wget https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data

--2026-05-19 13:05:53--  https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘iris.data.2’

iris.data.2             [ <=>                ]   4.44K  --.-KB/s    in 0s      

2026-05-19 13:05:54 (50.3 MB/s) - ‘iris.data.2’ saved [4551]



## 1. load data

In [103]:

import pandas as pd

df_pd = pd.read_csv("iris.data",
    names=['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid', 'species']
)

df = spark.createDataFrame(df_pd)
df.show(10)
print("total count:", df.count())


+---------+---------+---------+---------+-----------+
|sepal_len|sepal_wid|petal_len|petal_wid|    species|
+---------+---------+---------+---------+-----------+
|      5.1|      3.5|      1.4|      0.2|Iris-setosa|
|      4.9|      3.0|      1.4|      0.2|Iris-setosa|
|      4.7|      3.2|      1.3|      0.2|Iris-setosa|
|      4.6|      3.1|      1.5|      0.2|Iris-setosa|
|      5.0|      3.6|      1.4|      0.2|Iris-setosa|
|      5.4|      3.9|      1.7|      0.4|Iris-setosa|
|      4.6|      3.4|      1.4|      0.3|Iris-setosa|
|      5.0|      3.4|      1.5|      0.2|Iris-setosa|
|      4.4|      2.9|      1.4|      0.2|Iris-setosa|
|      4.9|      3.1|      1.5|      0.1|Iris-setosa|
+---------+---------+---------+---------+-----------+
only showing top 10 rows
total count: 150


## 2. prepare data

In [104]:

from pyspark.ml.feature import StringIndexer, VectorAssembler
# transfer into numeric data
indexer = StringIndexer(inputCol="species", outputCol="label")

featue_cols = ['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid']
assembler = VectorAssembler(inputCols=featue_cols, outputCol="features")

df_prep = indexer.fit(df).transform(df)
df_prep = assembler.transform(df_prep)

df_prep.select("features", "label", "species").show(5)


+-----------------+-----+-----------+
|         features|label|    species|
+-----------------+-----+-----------+
|[5.1,3.5,1.4,0.2]|  0.0|Iris-setosa|
|[4.9,3.0,1.4,0.2]|  0.0|Iris-setosa|
|[4.7,3.2,1.3,0.2]|  0.0|Iris-setosa|
|[4.6,3.1,1.5,0.2]|  0.0|Iris-setosa|
|[5.0,3.6,1.4,0.2]|  0.0|Iris-setosa|
+-----------------+-----+-----------+
only showing top 5 rows


## 3. train and test dataset split

In [105]:

train, test = df_prep.randomSplit([0.8, 0.2],seed=12333)
print("train:  " , train.count(),"test: ", test.count())
print("train datset composition ")
train.groupBy("species").count().show()

train:   128 test:  22
train datset composition 
+---------------+-----+
|        species|count|
+---------------+-----+
|    Iris-setosa|   43|
|Iris-versicolor|   43|
| Iris-virginica|   42|
+---------------+-----+



## 4. evaluate on test dataset

In [75]:


from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def my_evaluate(preds):
    #acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",metricName="accuracy")
    acc_eval=MulticlassClassificationEvaluator(metricName="accuracy")
    acc=acc_eval.evaluate(preds)

    f1_eval=MulticlassClassificationEvaluator(metricName="f1")
    f1=f1_eval.evaluate(preds)

    prec_eval=MulticlassClassificationEvaluator(metricName="weightedPrecision")
    prec = prec_eval.evaluate(preds)

    print("\n result:" )
    print("  accuracy:   " , round(acc,3))
    print("  f1 score:   " , round(f1,3))
    print("  precision:   " , round(prec,3))

    return {'acc': acc, 'f1': f1, 'prec': prec}



## 5. model 1 --- decision tree

In [76]:


from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42)

#param grid
dt_grid = ParamGridBuilder().addGrid(dt.maxDepth, [3, 5, 7]).build()

#cross validation
dt_cv = CrossValidator(
    estimator=dt,
    estimatorParamMaps=dt_grid,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    ),
    numFolds=3,
    seed=1111
)

# fit model
dt_model = dt_cv.fit(train)

dt_best = dt_model.bestModel
print("best maxDepth: " , str(dt_best.getMaxDepth()))

print("Decision Tree:")
dt_pred = dt_model.transform(test)
dt_result = my_evaluate(dt_pred)



best maxDepth:  5
Decision Tree:

 result:
  accuracy:    0.909
  f1 score:    0.908
  precision:    0.929


In [77]:
dt_pred.show(10)

+---------+---------+---------+---------+---------------+-----+-----------------+--------------+-------------+----------+
|sepal_len|sepal_wid|petal_len|petal_wid|        species|label|         features| rawPrediction|  probability|prediction|
+---------+---------+---------+---------+---------------+-----+-----------------+--------------+-------------+----------+
|      4.4|      3.2|      1.3|      0.2|    Iris-setosa|  0.0|[4.4,3.2,1.3,0.2]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|      4.7|      3.2|      1.6|      0.2|    Iris-setosa|  0.0|[4.7,3.2,1.6,0.2]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|      4.8|      3.0|      1.4|      0.3|    Iris-setosa|  0.0|[4.8,3.0,1.4,0.3]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|      5.0|      3.5|      1.3|      0.3|    Iris-setosa|  0.0|[5.0,3.5,1.3,0.3]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|      5.2|      3.5|      1.5|      0.2|    Iris-setosa|  0.0|[5.2,3.5,1.5,0.2]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|      5.5|      4.2|   

## 6. model 2 --- mlp

In [78]:


from pyspark.ml.classification import MultilayerPerceptronClassifier

# layers: 4 input, hidden 5 and 4, output 3
layers = [4, 2, 3]

mlp = MultilayerPerceptronClassifier(
    maxIter=50,
    layers=layers,
    seed=1111,
    featuresCol="features",
    labelCol="label"
)

# fit model
mlp_model = mlp.fit(train)

print("MLP:")
mlp_preds = mlp_model.transform(test)
mlp_result = my_evaluate(mlp_preds)




MLP:

 result:
  accuracy:    1.0
  f1 score:    1.0
  precision:    1.0


## 7. model 3 --- logistic regression

In [79]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",

    maxIter=100
)

# param grid
lr_grid = ParamGridBuilder() \
    .addGrid(lr.regParam,  [0.1, 0.5 ,1.0])\
    .build()

# cross validation
lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_grid,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    ),
    numFolds=3,
    seed=42
)


lr_model = lr_cv.fit(train)
lr_best = lr_model.bestModel

print("Logistic Regression")
lr_preds = lr_model.transform(test)
lr_result = my_evaluate(lr_preds)

Logistic Regression

 result:
  accuracy:    0.909
  f1 score:    0.907
  precision:    0.927


##

## 8. compare

From the results, MLP got 1.0 on all three metrics, so it performed the best.Decision Tree and Logistic were pretty close, both around 90% accuracy. If we try more grid search combinations for DT and LR, they might get better result.

Decision Tree

Strength:

* easy to understand, can see the rules,can handle non-linear relationships

limitiation:

* easy to overfit, especially when the tree is too deep

Multilayer Perceptron

Good:

* strong fitting ability, can learn complex boundaries,got perfect scores in this experiment

Bad:

* hard to interpret, can't really know what the model learned,takes longer to train

Logistic Regression

Good:

* simple model, trains fast

Bad:

* it's a linear model, not great for non-linear data

In [93]:
print("\nmodel     acc        f1        precision")
print("DT        "+str(round(dt_result['acc'], 4))+"     " + str(round(dt_result['f1'], 4)) +"     " + str(round(dt_result['prec'], 4)))
print("MLP       "+str(round(mlp_result['acc'], 4))+ "        " + str(round(mlp_result['f1'], 4))+ "        " + str(round(mlp_result['prec'], 4)))
print("Logistic  "+str(round(lr_result['acc'], 4))+"     " + str(round(lr_result['f1'], 4)) +"     " + str(round(lr_result['prec'], 4)))


model     acc        f1        precision
DT        0.9091     0.9083     0.9293
MLP       1.0        1.0        1.0
Logistic  0.9091     0.9066     0.9273


In [81]:


preds = mlp_model.transform(test)
preds.show(10)

correct = preds.filter("label = prediction").count()
wrong = preds.filter("label != prediction").count()
print("\n correct: %d, wrong: %d" % (correct, wrong))





+---------+---------+---------+---------+---------------+-----+-----------------+--------------------+--------------------+----------+
|sepal_len|sepal_wid|petal_len|petal_wid|        species|label|         features|       rawPrediction|         probability|prediction|
+---------+---------+---------+---------+---------------+-----+-----------------+--------------------+--------------------+----------+
|      4.4|      3.2|      1.3|      0.2|    Iris-setosa|  0.0|[4.4,3.2,1.3,0.2]|[29.7783545451229...|[0.99999999999899...|       0.0|
|      4.7|      3.2|      1.6|      0.2|    Iris-setosa|  0.0|[4.7,3.2,1.6,0.2]|[29.3530512606298...|[0.99999999999835...|       0.0|
|      4.8|      3.0|      1.4|      0.3|    Iris-setosa|  0.0|[4.8,3.0,1.4,0.3]|[29.1028383747530...|[0.99999999999780...|       0.0|
|      5.0|      3.5|      1.3|      0.3|    Iris-setosa|  0.0|[5.0,3.5,1.3,0.3]|[31.0588939084957...|[0.99999999999977...|       0.0|
|      5.2|      3.5|      1.5|      0.2|    Iris-setos

In [82]:

spark.stop()